# 03 - ColBERT: Dense Retrieval via Late Interaction

This notebook walks you through:

1. Installing the extra dependencies needed for ColBERT
2. Indexing all passages (encoding + storing multi-vector embeddings in PostgreSQL)
3. Running search queries with ColBERT (MaxSim)
4. Evaluating retrieval quality against the `qrels` ground truth

> **Prerequisite:** You must have run `01_data_preparation.ipynb` first so the
> `passages`, `queries`, and `qrels` tables are populated.

---

## Rappel des équations clés

### Encodeur de requête (§3.1)
$$f_Q(q) = \text{Normalize}\left(\mathbf{W} \cdot \text{BERT}(\texttt{[Q]}\; q_0\; q_1\; \ldots\; q_l\; \texttt{\#} \ldots \texttt{\#})\right)$$

### Encodeur de document (§3.1)
$$f_D(d) = \text{Filter}\left(\text{Normalize}\left(\mathbf{W} \cdot \text{BERT}(\texttt{[D]}\; d_0\; d_1\; \ldots\; d_n)\right)\right)$$

### Score de pertinence — Late Interaction MaxSim (§3.2)
$$S_{q,d} = \sum_{i \in |\hat{q}|} \max_{j \in |\hat{d}|} \; \mathbf{E}_{q_i} \cdot \mathbf{E}_{d_j}^T$$

## 0) Setup – project root on `sys.path`

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## 1) Install extra dependencies

ColBERT requires `transformers` and `torch` on top of the base project requirements.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(project_root / "requirements.txt")],
    check=True,
)

## 2) Index passages with ColBERT

This step encodes every passage in the `passages` table and stores the resulting multi-vector embeddings (`vector[]`) in the `colbert` table.

Each passage produces one embedding per token (after punctuation filtering), with each embedding having dimension 128.

**Estimated time:** Slower than SPLADE on CPU since each passage goes through BERT individually.

You can re-run safely — already-indexed passages are skipped.

In [ ]:
from src.colbert.indexer import index_passages

# Adjust batch size depending on your machine's RAM / VRAM
index_passages(batch_size=64)

### Verify indexing

In [ ]:
import pandas as pd
import warnings
from src.database.connection import get_connection

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy.*")

conn = get_connection()
try:
    count = pd.read_sql_query("SELECT COUNT(*) AS indexed FROM colbert", conn)
    print(f"Indexed passages: {count['indexed'][0]}")

    print("\nSample ColBERT entry (first passage):")
    sample = pd.read_sql_query(
        "SELECT passage_id, array_length(embedding, 1) AS num_vectors FROM colbert ORDER BY passage_id LIMIT 5",
        conn
    )
    display(sample)
finally:
    conn.close()

## 3) Search with ColBERT

ColBERT uses **brute-force MaxSim** scoring:
- Encode query → (N_q, 128) matrix
- For each indexed passage, load its (N_d, 128) matrix
- Compute MaxSim: for each query token, find the max similarity with any document token, then sum

In [ ]:
from src.colbert.search import search_bruteforce

query = "what is the speed of light"

print(f"Query: '{query}'\n")
results = search_bruteforce(query, top_k=10)

for i, r in enumerate(results, 1):
    print(f"[{i}] Score={r['score']:.4f}  Passage#{r['passage_id']}")
    print(f"    {r['text'][:150]}...\n")

In [ ]:
# Try your own queries
for q in ["how old is the earth", "who wrote hamlet", "symptoms of diabetes"]:
    results = search_bruteforce(q, top_k=3)
    print(f"\nQuery: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r['score']:.4f} — {r['text'][:120]}...")

## 4) Evaluation — MRR@10

Compute **Mean Reciprocal Rank @ 10** using the ground-truth `qrels` table.

In [ ]:
import random
from src.database.connection import get_connection
from src.colbert.search import search_bruteforce
from src.colbert.encoder import ColBERTEncoder

conn = get_connection()
cursor = conn.cursor()

# Get queries that have at least one relevant passage (relevance = 1)
cursor.execute("""
    SELECT DISTINCT q.id, q.text
    FROM queries q
    JOIN qrels qr ON qr.query_id = q.id
    WHERE qr.relevance = 1
""")
eval_queries = cursor.fetchall()

# Sample a subset for faster evaluation
SAMPLE_SIZE = 20
if len(eval_queries) > SAMPLE_SIZE:
    random.seed(42)
    eval_queries = random.sample(eval_queries, SAMPLE_SIZE)

print(f"Evaluating on {len(eval_queries)} queries...\n")

encoder = ColBERTEncoder()  # load once
reciprocal_ranks = []

for idx, (query_id, query_text) in enumerate(eval_queries):
    # Relevant passage IDs for this query
    cursor.execute(
        "SELECT passage_id FROM qrels WHERE query_id = %s AND relevance = 1",
        (query_id,)
    )
    relevant_ids = {row[0] for row in cursor.fetchall()}

    # Run ColBERT search
    results = search_bruteforce(
        query_text, top_k=10, conn=conn, encoder=encoder, log_search=False
    )

    # Compute reciprocal rank
    rr = 0.0
    for rank, r in enumerate(results, 1):
        if r["passage_id"] in relevant_ids:
            rr = 1.0 / rank
            break
    reciprocal_ranks.append(rr)

    if (idx + 1) % 5 == 0:
        current_mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
        print(f"  [{idx+1}/{len(eval_queries)}] Running MRR@10 = {current_mrr:.4f}")

mrr_at_10 = sum(reciprocal_ranks) / len(reciprocal_ranks)
print(f"\n{'='*50}")
print(f"Final MRR@10 = {mrr_at_10:.4f}  ({len(eval_queries)} queries)")
print(f"{'='*50}")

cursor.close()
conn.close()

## 5) Architecture Summary

| Component | Detail |
|---|---|
| **Backbone** | `bert-base-uncased` (110M parameters) |
| **Projection** | Linear 768 → 128, no bias |
| **Query special token** | `[Q]` → `[unused0]` |
| **Document special token** | `[D]` → `[unused1]` |
| **Query padding** | `[MASK]` (represents `#` from the paper) |
| **Document filtering** | Punctuation tokens removed |
| **Normalisation** | L2 per token |
| **Scoring** | MaxSim (Late Interaction) |
| **Equation** | $S_{q,d} = \sum_i \max_j \; E_{q_i} \cdot E_{d_j}^T$ |
| **Storage** | PostgreSQL `vector[]` via pgvector |

## Notes

- The first run downloads `bert-base-uncased` from Hugging Face (~440 MB).
- ColBERT indexing is slower than SPLADE because each passage produces multiple dense vectors instead of one sparse vector.
- Brute-force search computes MaxSim against all indexed passages. For large collections, consider approximate nearest-neighbor techniques.
- You can re-run indexing at any time; already-indexed passages are skipped automatically.